In [1]:
import re
import numpy as np
import pandas as pd
import os
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../notebooks")
print(os.listdir(DATA_DIR))
members = pd.read_csv(
    DATA_DIR / "members.csv",
    parse_dates=["join_date"]
)

renewals = pd.read_csv(
    DATA_DIR / "renewal_opportunities.csv",
    parse_dates=["expiry_date", "renewal_deadline"]
)

events = pd.read_csv(
    DATA_DIR / "member_events.csv",
    parse_dates=["event_time"]
)

# Date columns supplied for the exercises.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month

['weather2.csv', 'member_events.csv', 'members.csv', 'weather.csv', 'weather3.csv', 'renewal_opportunities.csv', 'DATA_DICTIONARY.md', 'Kalpa_Plus_Student.ipynb']


In [2]:
DATA_DIR

PosixPath('../notebooks')

1
Using members, group by acquisition_channel.

Write a function that receives the annual_fee Series and returns the difference between its highest and lowest values. Produce one row per channel with an output column named fee_range.

In [3]:
members.head()

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


In [4]:
def get_fee_range(annual_fees):
    return annual_fees.max() - annual_fees.min()

curr_members = members.copy()
fee_range = (
    curr_members.groupby('acquisition_channel')['annual_fee']
    .apply(get_fee_range)
    .reset_index(name="fee_range")
)
fee_range

,acquisition_channel,fee_range
0,APP,1000
1,STORE,1000
2,WEB,1000


2.
Using members, group by city.

The function should receive the group DataFrame. Among members who were "PAID" on 9 October 2025, calculate the percentage who are "EXPIRED" on 9 October 2026.

Return one scalar per city, named expired_previous_paid_pct. If a city has no previously paid members, return NaN. Members previously marked "NOT_JOINED" or "EXPIRED" must not enter the denominator.

In [5]:
def get_expired_percentage(city_members):
    previously_paid = city_members['status_2025_10_09'] == "PAID"
    denominator = previously_paid.sum()

    if denominator == 0:
        return np.nan

    expired_now = city_members['status_2026_10_09'] == "EXPIRED"
    return (expired_now & previously_paid).sum() / denominator * 100

In [6]:
curr_members = members.copy()
expired_previous_paid_pct = (
    curr_members.groupby('city')[['status_2025_10_09', 'status_2026_10_09']]
    .apply(get_expired_percentage)
    .reset_index(name="expired_previous_paid_pct")
)
expired_previous_paid_pct

,city,expired_previous_paid_pct
0,Ahmedabad,23.344371
1,Bengaluru,22.067901
2,Delhi,25.000000
3,Gurugram,24.267101
4,Hyderabad,24.344023
5,Jaipur,25.077399
6,Mumbai,24.644550
7,Pune,29.093800


3
Using members, group by acquisition_channel.

Write a function that receives the group DataFrame and returns the mean annual_fee of members currently marked "PAID". Produce current_paid_average_fee, with one row per acquisition channel.

In [7]:
def get_current_paid_average_fee(channel_members):
    currently_paid = channel_members.loc[
        channel_members['status_2026_10_09'] == "PAID", 'annual_fee'
    ]

    if currently_paid.empty:
        return np.nan

    return currently_paid.mean()

curr_members = members.copy()
current_paid_average_fee = (
    curr_members.groupby('acquisition_channel')[['status_2026_10_09', 'annual_fee']]
    .apply(get_current_paid_average_fee)
    .reset_index(name="current_paid_average_fee")
)
current_paid_average_fee

,acquisition_channel,current_paid_average_fee
0,APP,1803.505097
1,STORE,1792.863440
2,WEB,1789.799562


4
Using members, group by city.

Write a function that returns a named summary Series containing:
- registered_members: number of rows in the group.
- currently_paid: number currently marked "PAID".
- current_paid_pct: currently paid members as a percentage of all registered members in that city.

The final output should have one row per city and these three columns. This percentage is a billing-snapshot measure, not a renewal rate.

In [8]:
def summarize_city_members(city_members):
    currently_paid = city_members['status_2026_10_09'] == "PAID"
    registered_members = len(city_members)
    paid_count = currently_paid.sum()

    
    city_members["registered_members"]= registered_members
    city_members["currently_paid"]= paid_count
    city_members["current_paid_pct"]= round(paid_count / registered_members * 100,2)
    

city_paid_summary = (
    members.groupby('city')[['status_2026_10_09']]
    .apply(summarize_city_members)
    .reset_index()
)
city_paid_summary

,index,status_2026_10_09


5
Using members, group by city and pass the annual_fee Series to your function.

Return a Series showing each member's annual fee as a percentage of the sum of listed annual fees in that city. Assign it back to members as city_fee_share_pct.

Preserve every original row index. This is a share of listed fees, not a share of collected revenue. If a total is zero, return zero for every row in that group.

In [9]:
def get_city_fee_share(annual_fees):
    total_fees = annual_fees.sum()
    if total_fees == 0:
        return annual_fees * 0

    return annual_fees / total_fees * 100

curr_members = members.copy()
city_fee_share = (
    members.groupby('city', group_keys=False)['annual_fee']
    .apply(get_city_fee_share)
)
members.loc[city_fee_share.index, 'city_fee_share_pct'] = city_fee_share
curr_members

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023
...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023


6. Using members, group by city and pass the join_date Series to your function.

Return the number of whole days between each member's joining date and the earliest joining date in that city. Assign the returned Series as days_after_city_first_join.

The earliest joiner in each city should receive zero. Preserve the original indexes.

In [10]:
members.groupby('city')[['join_date','city']].transform('min')

,join_date,city
0,2023-09-01,Gurugram
1,2023-09-01,Bengaluru
2,2023-09-01,Bengaluru
3,2023-09-01,Delhi
4,2023-09-01,Jaipur
...,...,...
8575,2023-09-01,Hyderabad
8576,2023-09-01,Bengaluru
8577,2023-09-01,Pune
8578,2023-09-01,Gurugram


In [11]:
curr_members = members.copy()
def get_whole_days(join_dates):
    # print(join_dates)
    return (join_dates - min(join_dates)).dt.days


curr_members["days_after_city_first_join"] = curr_members.groupby('city')['join_date'].transform(get_whole_days)
print(curr_members.shape)
display(curr_members)

(8580, 11)


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,city_fee_share_pct,days_after_city_first_join
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,0.079970,21
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,0.128952,734
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,0.077350,12
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,0.078587,764
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,0.072860,25
...,...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,0.073305,891
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,0.077350,5
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,0.080291,922
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,0.133319,7


7. Using members, group by city.

For each city, calculate the mean annual fee among currently paid members. Your function should return a Series with one value for every original member row:

For a currently paid member, return their fee minus that city's mean fee among currently paid members.
For every other member, return NaN.
Assign the result as paid_fee_difference. Do not remove unpaid rows from the returned Series.

In [12]:
curr_members = members.copy()

def avg_annual_fee(curr_members):
    is_paid = curr_members['status_2026_10_09'] == "PAID"
    paid_members_mean = curr_members.loc[is_paid, 'annual_fee'].mean()
    res = curr_members['annual_fee']- paid_members_mean
    # print(res)

    return res.where(is_paid,np.nan)

curr_members["paid_fee_difference"] = (
    curr_members
    .groupby('city', group_keys=False)[['status_2026_10_09', 'annual_fee']]
    .apply(avg_annual_fee)
)
# curr_members["paid_fee_difference"]
curr_members

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,city_fee_share_pct,paid_fee_difference
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,0.079970,-283.443709
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,0.128952,677.042802
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,0.077350,-322.957198
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,0.078587,-287.798408
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,0.072860,-320.051414
...,...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,0.073305,-301.647655
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,0.077350,-322.957198
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,0.080291,-295.128940
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,0.133319,716.556291


8. Using members, group by city and send the group DataFrame to your function.

Return a DataFrame with the same row indexes and two columns:

city_paid_member_count: the number of currently paid members in the city, repeated on every member row.
paid_fee_share_pct: each currently paid member's share of that city's total listed fees among currently paid members. Use zero for members not currently paid and for a group with no paid fee total.
Assign both returned columns back to the original rows. Explain why summing city_paid_member_count over those rows would overcount members.

In [13]:
curr_members = members.copy()

def payment_details(df):
    is_paid = df['status_2026_10_09'] == "PAID"
    # paid_members = curr_members.loc[is_paid, 'annual_fee']
    # paid_members_count = curr_members.loc[is_paid].count()
    paid_members_count = is_paid.sum()
    paid_members_sum = df.loc[is_paid, 'annual_fee'].sum()
    if paid_members_sum:
        fee_share = (df['annual_fee'] / paid_members_sum * 100).where(is_paid, 0)
    else:
        fee_share = pd.Series(0,index=df.index)

    return pd.DataFrame({"city_paid_member_count": paid_members_count,
                         "paid_fee_share_pct": fee_share})



res = curr_members.groupby('city', group_keys=False)[['status_2026_10_09', 'annual_fee']].apply(payment_details)
print(res)
curr_members[["city_paid_member_count","paid_fee_share_pct"]] = res[["city_paid_member_count","paid_fee_share_pct"]]

curr_members


      city_paid_member_count  paid_fee_share_pct
0                        755            0.111388
1                        771            0.177899
2                        771            0.106711
3                        754            0.111264
4                        778            0.105920
...                      ...                 ...
8575                     789            0.105511
8576                     771            0.106711
8577                     698            0.119700
8578                     755            0.185696
8579                     771            0.177899

[8580 rows x 2 columns]


,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year,city_fee_share_pct,city_paid_member_count,paid_fee_share_pct
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023,0.079970,755,0.111388
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025,0.128952,771,0.177899
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023,0.077350,771,0.106711
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025,0.078587,754,0.111264
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023,0.072860,778,0.105920
...,...,...,...,...,...,...,...,...,...,...,...,...
8575,MP008500,2026-02-08,PLUS_STANDARD,1499,APP,Hyderabad,NOT_JOINED,PAID,2026,0.073305,789,0.105511
8576,MP002065,2023-09-06,PLUS_STANDARD,1499,APP,Bengaluru,PAID,PAID,2023,0.077350,771,0.106711
8577,MP007986,2026-03-11,PLUS_STANDARD,1499,WEB,Pune,NOT_JOINED,PAID,2026,0.080291,698,0.119700
8578,MP002304,2023-09-08,PLUS_PREMIUM,2499,WEB,Gurugram,PAID,PAID,2023,0.133319,755,0.185696


**Explanation for Question 8:**
Summing `city_paid_member_count` over all member rows would sum the group-level paid member count across every member in that city (both paid and unpaid). Because the paid member count is repeated on every member row, summing the column multiplies the paid member count by the total registered members in that city (e.g., $755 \times 1055 = 796,525$), resulting in massive overcounting.

9.
Using renewals, group by renewal_number.

Write a function receiving the offered_discount Series. Return the percentage of records with a discount greater than zero, rounded to two decimal places.

Produce one row per renewal number, with a column named discount_offer_pct.

In [14]:
renewals.head()

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


In [15]:
curr_renewals = renewals.copy()
def percentage_records(ds):
    is_discount = ds > 0
    total_records = ds.count()
    discounted_users_count = is_discount.sum()
    return round(discounted_users_count/total_records * 100,2)



# curr_renewals['discount_offer_pct'] = renewals.groupby("renewal_number")["offered_discount"].apply(percentage_records)
result = (curr_renewals.groupby("renewal_number")["offered_discount"].apply(percentage_records)
 .reset_index(name="discount_offer_pct"))

display(result)

,renewal_number,discount_offer_pct
0,1,7.88
1,2,10.82
2,3,11.72


10.
Using renewals, group by renewal_number and pass the group DataFrame.

Among records where offered_discount > 0, calculate the mean of renewal_fee - offered_discount.

Return one scalar per group, named average_discounted_offer. This is an average offered price, not average cash received.

In [16]:
curr_renewals = renewals.copy()

def get_average_discounted_offer(df):
    is_discount = df["offered_discount"] > 0
    if not is_discount.any():
        return np.nan
    return (df.loc[is_discount, 'renewal_fee'] - df.loc[is_discount, 'offered_discount']).mean()

average_discounted_offer = (
    curr_renewals.groupby("renewal_number")[["offered_discount", "renewal_fee"]]
    .apply(get_average_discounted_offer)
    .reset_index(name="average_discounted_offer")
)

average_discounted_offer

,renewal_number,average_discounted_offer
0,1,1653.174950
1,2,1596.010870
2,3,1617.720379


### 11

Using `renewals`, group by `expiry_year`.

Write a function receiving the group DataFrame. Count records whose `renewal_deadline` is before 10 October 2026: these windows have finished by the end of 9 October.

Return one scalar named `finished_windows` per year. Do not infer successful renewal from this table.

In [17]:
curr_renewals = renewals.copy()

def count_finished_windows(df):
    return (df['renewal_deadline'] < pd.Timestamp("2026-10-10")).sum()

finished_windows = (
    curr_renewals.groupby('expiry_year')[['renewal_deadline']]
    .apply(count_finished_windows)
    .reset_index(name="finished_windows")
)

finished_windows

,expiry_year,finished_windows
0,2024,2500
1,2025,4000
2,2026,4480


### 12

Using `renewals`, group by `expiry_year` and return a named summary Series containing:

- `members_due`: number of records.
- `finished_windows`: number whose deadline is before 10 October 2026.
- `open_windows`: number whose deadline is on or after 10 October 2026.

Produce one summary row per year. Check that finished plus open windows equals members due in every group.

In [18]:
curr_renewals = renewals.copy()

def summarize_renewal_windows(df):
    members_due = len(df)
    finished = (df['renewal_deadline'] < pd.Timestamp("2026-10-10")).sum()
    open_win = (df['renewal_deadline'] >= pd.Timestamp("2026-10-10")).sum()
    return pd.Series({
        'members_due': members_due,
        'finished_windows': finished,
        'open_windows': open_win
    })

renewal_window_summary = (
    curr_renewals.groupby('expiry_year')[['renewal_deadline']]
    .apply(summarize_renewal_windows)
    .reset_index()
)

# Verification check: finished plus open windows equals members due in every group
assert (renewal_window_summary['finished_windows'] + renewal_window_summary['open_windows'] == renewal_window_summary['members_due']).all()
print("Verification passed: finished_windows + open_windows == members_due for all groups.")

renewal_window_summary

Verification passed: finished_windows + open_windows == members_due for all groups.


,expiry_year,members_due,finished_windows,open_windows
0,2024,2500,2500,0
1,2025,4000,4000,0
2,2026,5080,4480,600


### 13

Using `renewals`, group by `renewal_number` and pass the `renewal_fee` Series.

Return a Boolean Series indicating whether each fee is strictly greater than its group's median fee. Assign it back as `above_group_median_fee`.

Values equal to the median should receive `False`.

In [19]:
curr_renewals = renewals.copy()

def is_above_group_median(fee_series):
    return fee_series > fee_series.median()

curr_renewals['above_group_median_fee'] = (
    curr_renewals.groupby('renewal_number', group_keys=False)['renewal_fee']
    .apply(is_above_group_median)
)

curr_renewals

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month,above_group_median_fee
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9,True
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9,True
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9,False
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9,False
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9,False
...,...,...,...,...,...,...,...,...,...,...
11575,RN003972,MP001840,2,2025-09-04,2025-09-11,2499,0,2025,9,True
11576,RN000572,MP002177,1,2024-09-03,2024-09-10,1499,0,2024,9,False
11577,RN009390,MP003830,2,2026-09-10,2026-09-17,1499,0,2026,9,False
11578,RN010994,MP005933,1,2026-10-07,2026-10-14,1499,0,2026,10,False


### 14

Using `renewals`, group by both `expiry_year` and `renewal_number`.

Write a function receiving the group DataFrame. Calculate each row's offered price as `renewal_fee - offered_discount`. Return a Series showing the difference between that row's offered price and its group's mean offered price.

Assign it back as `offer_price_difference`, keeping every original index.

In [20]:
curr_renewals = renewals.copy()

def get_offer_price_diff(df):
    offered_price = df['renewal_fee'] - df['offered_discount']
    return offered_price - offered_price.mean()

curr_renewals['offer_price_difference'] = (
    curr_renewals.groupby(['expiry_year', 'renewal_number'], group_keys=False)[['renewal_fee', 'offered_discount']]
    .apply(get_offer_price_diff)
)

curr_renewals

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month,offer_price_difference
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9,721.700000
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9,711.083333
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9,-292.820000
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9,-290.904255
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9,-278.300000
...,...,...,...,...,...,...,...,...,...,...
11575,RN003972,MP001840,2,2025-09-04,2025-09-11,2499,0,2025,9,709.250000
11576,RN000572,MP002177,1,2024-09-03,2024-09-10,1499,0,2024,9,-292.820000
11577,RN009390,MP003830,2,2026-09-10,2026-09-17,1499,0,2026,9,-270.857143
11578,RN010994,MP005933,1,2026-10-07,2026-10-14,1499,0,2026,10,-290.904255


### 15

Using `renewals`, group by `renewal_number`.

Your function should return a DataFrame with the same original indexes containing:

- `net_offer_price`: renewal fee minus offered discount.
- `below_group_mean_offer`: whether that row's net offer price is strictly below its group's mean net offer price.

Assign both columns back to the original renewal rows.

In [8]:
curr_renewals = renewals.copy()

def get_net_offer_summary(df):
    net_offer = df['renewal_fee'] - df['offered_discount']
    below_mean = net_offer < net_offer.mean()
    return pd.DataFrame({
        'net_offer_price': net_offer,
        'below_group_mean_offer': below_mean
    }, index=df.index)

res = (
    curr_renewals.groupby('renewal_number', group_keys=False)[['renewal_fee', 'offered_discount']]
    .apply(get_net_offer_summary)
)

curr_renewals[['net_offer_price', 'below_group_mean_offer']] = res[['net_offer_price', 'below_group_mean_offer']]

curr_renewals

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month,net_offer_price,below_group_mean_offer
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9,2499,False
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9,2499,False
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9,1499,True
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9,1499,True
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9,1499,True
...,...,...,...,...,...,...,...,...,...,...,...
11575,RN003972,MP001840,2,2025-09-04,2025-09-11,2499,0,2025,9,2499,False
11576,RN000572,MP002177,1,2024-09-03,2024-09-10,1499,0,2024,9,1499,True
11577,RN009390,MP003830,2,2026-09-10,2026-09-17,1499,0,2026,9,1499,True
11578,RN010994,MP005933,1,2026-10-07,2026-10-14,1499,0,2026,10,1499,True


### 16

Using `renewals`, group by `expiry_year`.

Return a DataFrame with one row for every original renewal record and two columns:

- `window_finished`: whether its deadline is before 10 October 2026.
- `finished_window_fee_share_pct`: its renewal fee as a percentage of the group's total renewal fees for finished windows only.

For an open window, its share should be zero. If the group's eligible fee total is zero, all shares should be zero. Assign both columns back. This share describes fees associated with eligible records, not actual renewals or cash received.

In [22]:
curr_renewals = renewals.copy()

def get_finished_window_fee_share(df):
    window_finished = df['renewal_deadline'] < pd.Timestamp("2026-10-10")
    eligible_total = df.loc[window_finished, 'renewal_fee'].sum()
    if eligible_total > 0:
        share = (df['renewal_fee'] / eligible_total * 100).where(window_finished, 0.0)
    else:
        share = pd.Series(0.0, index=df.index)
    return pd.DataFrame({
        'window_finished': window_finished,
        'finished_window_fee_share_pct': share
    }, index=df.index)

res = (
    curr_renewals.groupby('expiry_year', group_keys=False)[['renewal_deadline', 'renewal_fee']]
    .apply(get_finished_window_fee_share)
)

curr_renewals[['window_finished', 'finished_window_fee_share_pct']] = res[['window_finished', 'finished_window_fee_share_pct']]

curr_renewals

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month,window_finished,finished_window_fee_share_pct
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9,True,0.034689
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9,True,0.030953
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9,True,0.033153
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9,True,0.018567
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9,True,0.020808
...,...,...,...,...,...,...,...,...,...,...,...
11575,RN003972,MP001840,2,2025-09-04,2025-09-11,2499,0,2025,9,True,0.034689
11576,RN000572,MP002177,1,2024-09-03,2024-09-10,1499,0,2024,9,True,0.033153
11577,RN009390,MP003830,2,2026-09-10,2026-09-17,1499,0,2026,9,True,0.018567
11578,RN010994,MP005933,1,2026-10-07,2026-10-14,1499,0,2026,10,False,0.000000


### 17

Using `events`, first select only `"PAYMENT_ATTEMPT"` events. Group by `channel` and send the `outcome` Series to your function.

Return the percentage of failed attempts, rounded to two decimal places. Produce one row per channel with a column named `failure_pct`.

In [23]:
payment_events = events[events['event_type'] == "PAYMENT_ATTEMPT"].copy()

def get_failure_pct(outcome_series):
    total_attempts = len(outcome_series)
    if total_attempts == 0:
        return np.nan
    failed_attempts = (outcome_series == "FAILED").sum()
    return round((failed_attempts / total_attempts) * 100, 2)

channel_failure_pct = (
    payment_events.groupby('channel')['outcome']
    .apply(get_failure_pct)
    .reset_index(name="failure_pct")
)

channel_failure_pct

,channel,failure_pct
0,APP,39.26
1,WEB,23.21


### 18

Using payment-attempt rows from `events`, group by `channel` and pass the group DataFrame.

Return total payment-attempt cost divided by the number of successful payment events. Include failed-attempt costs in the numerator. Name the result `cost_per_successful_renewal`.

Return `NaN` if there are no successful payments. Do not divide by total attempts.

In [24]:
payment_events = events[events['event_type'] == "PAYMENT_ATTEMPT"].copy()

def get_cost_per_successful_renewal(df):
    successful_count = (df['outcome'] == "SUCCESS").sum()
    if successful_count == 0:
        return np.nan
    total_cost = df['event_cost'].sum()
    return total_cost / successful_count

cost_per_successful_renewal = (
    payment_events.groupby('channel')[['event_cost', 'outcome']]
    .apply(get_cost_per_successful_renewal)
    .reset_index(name="cost_per_successful_renewal")
)

cost_per_successful_renewal

,channel,cost_per_successful_renewal
0,APP,13.292791
1,WEB,12.604620


### 19

Using payment-attempt rows from `events`, group by `channel` and pass the `renewal_id` Series.

Return the percentage of distinct renewal IDs that have at least two attempts within that channel. Name the result `retry_pct`, rounded to two decimal places.

A renewal attempted in more than one channel can appear in both channel summaries. Explain why a high retry percentage does not necessarily mean those renewals ultimately failed.

In [25]:
payment_events = events[events['event_type'] == "PAYMENT_ATTEMPT"].copy()

def get_retry_pct(renewal_id_series):
    attempt_counts = renewal_id_series.value_counts()
    if len(attempt_counts) == 0:
        return np.nan
    retried_count = (attempt_counts >= 2).sum()
    return round((retried_count / len(attempt_counts)) * 100, 2)

retry_pct = (
    payment_events.groupby('channel')['renewal_id']
    .apply(get_retry_pct)
    .reset_index(name="retry_pct")
)

retry_pct

,channel,retry_pct
0,APP,20.32
1,WEB,10.65


**Explanation for Question 19:**
A high retry percentage does not necessarily mean those renewals ultimately failed. A retry simply indicates that an initial payment attempt did not succeed (for instance due to insufficient funds, gateway timeouts, expired cards, or transient bank declines) and required a subsequent attempt. In subscription systems, automated or customer-initiated retries frequently recover the payment on the second or third try. Therefore, retries capture process friction and retry recovery activity, not final customer churn or renewal loss.

### 20

Using payment-attempt rows from `events`, group by `channel`.

Return a named summary Series containing:

- `attempt_count`: number of payment attempts.
- `renewals_attempted`: number of distinct renewal IDs.
- `successful_payments`: number of successful payment events.
- `total_payment_cost`: cost of all attempts, including failures.

Produce a summary table with these four columns and one row per channel.

In [26]:
payment_events = events[events['event_type'] == "PAYMENT_ATTEMPT"].copy()

def summarize_payment_attempts(df):
    return pd.Series({
        'attempt_count': len(df),
        'renewals_attempted': df['renewal_id'].nunique(),
        'successful_payments': (df['outcome'] == "SUCCESS").sum(),
        'total_payment_cost': df['event_cost'].sum()
    })

payment_attempt_summary = (
    payment_events.groupby('channel')[['renewal_id', 'outcome', 'event_cost']]
    .apply(summarize_payment_attempts)
    .reset_index()
)

payment_attempt_summary

,channel,attempt_count,renewals_attempted,successful_payments,total_payment_cost
0,APP,7445,5822,4522,60110
1,WEB,5863,5296,4502,56746


### 21

Using all rows in `events`, group by `event_type` and send the `event_cost` Series to your function.

Return a Series showing each event's cost minus the median event cost for that event type. Assign it to the original rows as `cost_difference_from_median`.

Preserve the original indexes and retain negative differences.

In [27]:
curr_events = events.copy()

def get_cost_diff_from_median(cost_series):
    return cost_series - cost_series.median()

curr_events['cost_difference_from_median'] = (
    curr_events.groupby('event_type', group_keys=False)['event_cost']
    .apply(get_cost_diff_from_median)
)

curr_events

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,cost_difference_from_median
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,-24.0
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,-7.0
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,21.0
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,19.0
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,-21.0
...,...,...,...,...,...,...,...,...,...,...,...
152052,EV0148986,MP006126,RN011295,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.0
152053,EV0149338,MP005959,RN011311,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.0
152054,EV0149357,MP006087,RN011312,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.0
152055,EV0149742,MP005903,RN011330,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.0


### 22

Using all rows in `events`, group by `channel` and pass the group DataFrame.

Calculate the mean `amount` using only successful payment-attempt rows within that channel. Return a Series with one value per original event row:

- For every payment-attempt row, return its amount minus that successful-payment mean.
- For all non-payment rows, return `NaN`.
- If the channel has no successful payments, return `NaN` for every row.

Assign the result as `amount_difference`. Failed attempts receive a difference for comparison, but their amounts must not enter the successful-payment mean.

In [28]:
curr_events = events.copy()

def get_amount_difference(df):
    is_payment = df['event_type'] == "PAYMENT_ATTEMPT"
    is_successful_payment = is_payment & (df['outcome'] == "SUCCESS")
    
    if not is_successful_payment.any():
        return pd.Series(np.nan, index=df.index)
    
    successful_mean = df.loc[is_successful_payment, 'amount'].mean()
    diff = df['amount'] - successful_mean
    return diff.where(is_payment, np.nan)

curr_events['amount_difference'] = (
    curr_events.groupby('channel', group_keys=False)[['event_type', 'outcome', 'amount']]
    .apply(get_amount_difference)
)

curr_events

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,amount_difference
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,NaN
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,NaN
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,NaN
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,NaN
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,NaN
...,...,...,...,...,...,...,...,...,...,...,...
152052,EV0148986,MP006126,RN011295,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN
152053,EV0149338,MP005959,RN011311,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN
152054,EV0149357,MP006087,RN011312,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN
152055,EV0149742,MP005903,RN011330,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN


### 23

Using all rows in `events`, group by `renewal_id`.

Return a DataFrame with the same row indexes containing:

- `renewal_payment_cost`: total cost of payment attempts for that renewal, repeated on every event row.
- `renewal_failed_payment_cost`: total cost of failed payment attempts for that renewal, repeated on every event row.

Include benefit, reminder and support rows in the returned result. A renewal with no payment attempts should receive zero in both columns. Assign the results back without changing the original row count.

In [29]:
curr_events = events.copy()

def get_renewal_payment_costs(df):
    is_payment = df['event_type'] == "PAYMENT_ATTEMPT"
    if is_payment.any():
        pmt_df = df[is_payment]
        payment_cost = pmt_df['event_cost'].sum()
        failed_payment_cost = pmt_df.loc[pmt_df['outcome'] == "FAILED", 'event_cost'].sum()
    else:
        payment_cost = 0
        failed_payment_cost = 0
    return pd.DataFrame({
        'renewal_payment_cost': payment_cost,
        'renewal_failed_payment_cost': failed_payment_cost
    }, index=df.index)

res_23 = (
    curr_events.groupby('renewal_id', group_keys=False)[['event_type', 'outcome', 'event_cost']]
    .apply(get_renewal_payment_costs)
)

curr_events[['renewal_payment_cost', 'renewal_failed_payment_cost']] = res_23[['renewal_payment_cost', 'renewal_failed_payment_cost']]

curr_events

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,renewal_payment_cost,renewal_failed_payment_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,12,0
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,12,0
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,14,2
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,0,0
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
152052,EV0148986,MP006126,RN011295,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,4,4
152053,EV0149338,MP005959,RN011311,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,4,4
152054,EV0149357,MP006087,RN011312,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,4,4
152055,EV0149742,MP005903,RN011330,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,4,4


### 24

Using all rows in `events`, group by `event_type`.

Return a DataFrame with the original indexes containing:

- `type_cost_share_pct`: each row's share of its event type's total event cost. Use zero if total cost is zero.
- `above_type_average_cost`: whether each row costs strictly more than its event type's average.

Assign both returned columns back. For a group with a positive cost total, check that its cost shares sum to approximately 100%.

In [30]:
curr_events = events.copy()

def get_type_cost_share(df):
    costs = df['event_cost']
    total_cost = costs.sum()
    mean_cost = costs.mean()
    if total_cost > 0:
        share = (costs / total_cost) * 100
    else:
        share = pd.Series(0.0, index=df.index)
    above_avg = costs > mean_cost
    return pd.DataFrame({
        'type_cost_share_pct': share,
        'above_type_average_cost': above_avg
    }, index=df.index)

res_24 = (
    curr_events.groupby('event_type', group_keys=False)[['event_cost']]
    .apply(get_type_cost_share)
)

curr_events[['type_cost_share_pct', 'above_type_average_cost']] = res_24[['type_cost_share_pct', 'above_type_average_cost']]

# Check that cost shares sum to approximately 100% for each event type
share_sums = curr_events.groupby('event_type')['type_cost_share_pct'].sum()
print("Cost share sum per event type:\n", share_sums)

curr_events

Cost share sum per event type:
 event_type
BENEFIT_USED       100.0
PAYMENT_ATTEMPT    100.0
REMINDER_SENT      100.0
SUPPORT_CONTACT    100.0
Name: type_cost_share_pct, dtype: float64


,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,type_cost_share_pct,above_type_average_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,0.000221,False
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,0.000596,False
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,0.001215,True
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,0.001171,True
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,0.000287,False
...,...,...,...,...,...,...,...,...,...,...,...,...
152052,EV0148986,MP006126,RN011295,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.085179,False
152053,EV0149338,MP005959,RN011311,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.085179,False
152054,EV0149357,MP006087,RN011312,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.085179,False
152055,EV0149742,MP005903,RN011330,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,0.085179,False


### 25

Using `events`, select only payment-attempt rows, retaining their original indexes. Group by `renewal_id`.

Inside the function, sort the group's rows by `event_time`, using `event_id` to break any timestamp ties. Return a DataFrame containing:

- `chronological_attempt_number`: 1, 2, 3 and so on in timestamp order.
- `minutes_since_first_attempt`: elapsed minutes from the group's first attempt.

Keep the original event indexes even after sorting. Assign both returned columns to the corresponding rows in the full `events` DataFrame. Non-payment rows should remain `NaN` in these columns.

Explain why resetting the row index inside the function could place a result beside the wrong event.

In [31]:
curr_events = events.copy()
payment_attempts = curr_events[curr_events['event_type'] == "PAYMENT_ATTEMPT"].copy()

def compute_attempt_sequence(df):
    sorted_df = df.sort_values(by=['event_time', 'event_id'])
    attempt_num = np.arange(1, len(sorted_df) + 1)
    first_time = sorted_df['event_time'].iloc[0]
    elapsed_minutes = (sorted_df['event_time'] - first_time).dt.total_seconds() / 60.0
    return pd.DataFrame({
        'chronological_attempt_number': pd.Series(attempt_num, index=sorted_df.index),
        'minutes_since_first_attempt': elapsed_minutes
    }, index=sorted_df.index)

res_25 = (
    payment_attempts.groupby('renewal_id', group_keys=False)[['event_time', 'event_id']]
    .apply(compute_attempt_sequence)
)

curr_events[['chronological_attempt_number', 'minutes_since_first_attempt']] = np.nan
curr_events.loc[res_25.index, ['chronological_attempt_number', 'minutes_since_first_attempt']] = res_25

# Verification checks
assert curr_events.loc[curr_events['event_type'] != "PAYMENT_ATTEMPT", 'chronological_attempt_number'].isna().all()
assert curr_events.loc[curr_events['event_type'] == "PAYMENT_ATTEMPT", 'chronological_attempt_number'].notna().all()
print("Verification passed: Non-payment rows remain NaN, payment rows correctly updated.")

curr_events

Verification passed: Non-payment rows remain NaN, payment rows correctly updated.


,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost,chronological_attempt_number,minutes_since_first_attempt
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10,NaN,NaN
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27,NaN,NaN
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55,NaN,NaN
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53,NaN,NaN
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
152052,EV0148986,MP006126,RN011295,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN,NaN
152053,EV0149338,MP005959,RN011311,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN,NaN
152054,EV0149357,MP006087,RN011312,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN,NaN
152055,EV0149742,MP005903,RN011330,2026-10-09 14:00:00,SUPPORT_CONTACT,SUPPORT,LOGGED,FREE_DELIVERY_UNAVAILABLE,0,45,NaN,NaN


**Explanation for Question 25:**
Resetting the row index inside the function (such as with `.reset_index(drop=True)`) replaces the group's original DataFrame index labels with a standard zero-based integer index `0, 1, 2, ...`.

Because the rows are sorted chronologically by `event_time` (and tie-broken by `event_id`), the rows within the group are reordered relative to their original positions in the table. If the row index is reset, the connection to the original row labels is lost. When the grouped results are concatenated and assigned back by index, pandas will align rows by the newly generated integer index labels rather than the true original row positions, attaching attempt counts and elapsed minutes to completely wrong event records. Keeping the original index labels guarantees that every computed metric maps accurately back to its exact event row.